# HCP engagement

Per-HCP engagement snapshot: last visit, visits in last 90 days, open follow-ups, engagement tier. Pandas + a bit of Spark.

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.master('local[*]').getOrCreate()
DATA = '../data'
AS_OF = pd.Timestamp('2025-09-29')

In [ ]:
hcps = spark.read.parquet(DATA + '/hcps.parquet')
visits = spark.read.parquet(DATA + '/visits.parquet')
fu = pd.read_parquet(DATA + '/follow_ups.parquet')

In [ ]:
last = visits.groupBy('hcp_id').agg(F.max('visit_date').alias('last_visit'), F.count('*').alias('total_visits')).toPandas()
v = visits.toPandas()
v['visit_date'] = pd.to_datetime(v['visit_date'])
recent = v[v['visit_date'] >= AS_OF - pd.Timedelta(days=90)].groupby('hcp_id').size().reset_index(name='visits_90d')
open_fu = fu[fu['status'] == 'open'].groupby('hcp_id').size().reset_index(name='open_follow_ups')

In [ ]:
df = hcps.toPandas().merge(last, on='hcp_id', how='left').merge(recent, on='hcp_id', how='left').merge(open_fu, on='hcp_id', how='left')
df['last_visit'] = pd.to_datetime(df['last_visit'])
df['days_since_last_visit'] = (AS_OF - df['last_visit']).dt.days
for c in ['total_visits', 'visits_90d', 'open_follow_ups']:
    df[c] = df[c].fillna(0).astype(int)
def tier(r):
    if r['visits_90d'] >= 3: return 'high'
    if r['visits_90d'] >= 1: return 'medium'
    return 'low'
df['engagement_tier'] = df.apply(tier, axis=1)
df = df.sort_values('hcp_id').reset_index(drop=True)
df.head()

In [ ]:
df.to_parquet('expected_output/hcp_engagement.parquet', index=False)